ESERCIZIO

Inferenza Strutturata e Streaming

1) utilizza la libreria 'Transformers' per caricare un modello leggero (es. GPT2 o TinyLlama)
2) Implementa un ciclo di generazione che si interrompa non appena il modello genera il carattere punto esclamativo.
3) Configura un 'TextStreamer' per visualizzare i token sulla console in tempo reale.
4) Sfida: aggiungi un vincolo manuale che impedisca la generazione di numeri durante la sequenza.

Suggerimento: usa un 'LogitsProcessor' personalizzato per azzerare i logit dei token numerici.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, LogitsProcessor, LogitsProcessorList, StoppingCriteria, StoppingCriteriaList

# 1. CONFIGURAZIONE DEL MODELLO E DEL TOKENIZER
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. SFIDA: VINCOLO MANUALE ANTI-NUMERI (LogitsProcessor)
class AvoidNumbersProcessor(LogitsProcessor):
    """
    Questo processore implementa il Logit Masking. 
    Azzera la probabilità di ogni token che contenga una cifra (0-9).
    """
    def __call__(self, input_ids, scores):
        # Iteriamo sull'intero vocabolario per identificare i token numerici
        for token_id in range(len(scores[0])):
            token_text = tokenizer.decode([token_id])
            
            # Se il token contiene un numero, lo rendiamo impossibile da scegliere
            if any(char.isdigit() for char in token_text):
                scores[:, token_id] = float('-inf') 
        return scores

# 3. CRITERIO DI STOP: PUNTO ESCLAMATIVO (StoppingCriteria)
class StopAtExclamationCriteria(StoppingCriteria):
    """
    Interrompe la generazione non appena viene rilevato un punto esclamativo.
    Utile per controllare la terminazione e risparmiare risorse computazionali.
    """
    def __call__(self, input_ids, scores, **kwargs):
        # Decodifichiamo l'ultimo token prodotto nell'ultimo passo di generazione
        last_token = tokenizer.decode(input_ids[0][-1])
        
        # Restituiamo True per fermare il loop se troviamo '!'
        return "!" in last_token

# 4. PREPARAZIONE DELLA GENERAZIONE IN STREAMING
prompt = "Tell me a short story about the number 42:"
inputs = tokenizer(prompt, return_tensors="pt")

# TextStreamer visualizza i token sulla console in tempo reale (Emissione asincrona)
streamer = TextStreamer(tokenizer, skip_prompt=True)

# 5. COORDINAZIONE DEI COMPONENTI
# Inseriamo i nostri vincoli nelle liste dedicate
logits_processors = LogitsProcessorList([AvoidNumbersProcessor()])
stopping_criteria = StoppingCriteriaList([StopAtExclamationCriteria()])

print(f"Prompt iniziale: {prompt}")
print("-" * 30)
print("Risposta in streaming (Senza numeri e stop al primo '!'):")

# 6. ESECUZIONE DEL PROCESSO DI GENERAZIONE
with torch.no_grad():
    model.generate(
        **inputs,
        streamer=streamer,
        logits_processor=logits_processors,
        stopping_criteria=stopping_criteria,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4633.00it/s]


Prompt iniziale: Tell me a short story about the number 42:
------------------------------
Risposta in streaming (Senza numeri e stop al primo '!'):


The New York Times gave him a list. The Times asked him to list the places where he could see them. He did, and when he found the number in the newspaper, he went and called the local government, and the sheriff's office.

It was the first story he'd seen in a while, he said.

Then the news came and they were sending it to the sheriff's office, and he went out to walk back. When he went back to go collect the story he heard, "I got this picture of this guy in the backyard and this guy in the kitchen. There was about a dozen or more people in the house and all of a sudden everything started to go quiet. I was literally in
